# convtranspose-bn-activation-block composite — cx10: chain three ConvT+BN+ReLU generator blocks via nn.Sequential

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `module-composition`, `convtranspose-bn-activation-block`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F
from einops.layers.torch import Rearrange

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "convtranspose-bn-activation-block"
DD_ATOM_IDS = ["module-composition", "convtranspose-bn-activation-block"]
DD_SUBTOPICS = ["PyTorch: Module composition", "GAN: ConvT+BN+Activation block"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

A DCGAN generator is a STACK of ConvT+BN+ReLU blocks (the `convtranspose-bn-activation-block` atom), one per doubling of spatial size. Each block follows the same recipe:
`nn.Sequential(nn.ConvTranspose2d(stride=2, kernel=4, padding=1, bias=False), nn.BatchNorm2d, nn.ReLU)`.

Composing N of these blocks back-to-back is the `module-composition` atom in action: you wrap them in an OUTER `nn.Sequential` and the spatial size doubles N times.

**Why both atoms together.** A single block produces ONE doubling; stacking blocks produces the full 4 -> 8 -> 16 -> 32 upsampling pipeline that maps a 4x4 noise-derived feature map to a 32x32 image (or 64x64, or 128x128 with another block). The outer `Sequential` IS the composition; without it you'd be writing a custom forward.

**Anatomy (three-block stack: 4x4 -> 32x32).**
```python
def block(c_in, c_out):
    return nn.Sequential(
        nn.ConvTranspose2d(c_in, c_out, kernel_size=4, stride=2, padding=1, bias=False),
        nn.BatchNorm2d(c_out),
        nn.ReLU(inplace=True),
    )

generator = nn.Sequential(
    block(256, 128),  # 4x4 -> 8x8.
    block(128, 64),   # 8x8 -> 16x16.
    block(64, 32),    # 16x16 -> 32x32.
)
```

Note: `bias=False` on the ConvT because BN immediately follows (BN subtracts the mean, rendering the conv bias redundant).

### Composite Exercise — chain three ConvT+BN+ReLU generator blocks via nn.Sequential

**Atoms exercised together**: `module-composition`, `convtranspose-bn-activation-block`

Implement `cx10_make_generator_stack(channels)` — return an `nn.Sequential` of THREE ConvT+BN+ReLU blocks.

Inputs: `channels` is a list of 4 ints, e.g. `[256, 128, 64, 32]`. The k-th block transforms `channels[k] -> channels[k+1]` and DOUBLES the spatial dim.

Each block (you may write a helper) is:
```
nn.Sequential(
    nn.ConvTranspose2d(c_in, c_out, kernel_size=4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(c_out),
    nn.ReLU(inplace=True),
)
```

Then `nn.Sequential` the three blocks together and RETURN that outer Sequential. The outer Sequential contains 3 child modules (each itself a Sequential of 3 layers).

Test checks:
- Outer return is `nn.Sequential` with exactly 3 children.
- Each child is itself a Sequential of 3 layers: ConvTranspose2d, BatchNorm2d, ReLU.
- ConvT params: `kernel=4, stride=2, padding=1, bias=None` (bias=False).
- Channel chain matches `channels`.
- Running a `(1, channels[0], 4, 4)` input through gives `(1, channels[3], 32, 32)` — spatial dim doubled 3 times.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx10_make_generator_stack(channels):
    """Return nn.Sequential of THREE ConvT+BN+ReLU blocks with channel chain = channels."""
    raise NotImplementedError

def _test_cx10():
    channels = [256, 128, 64, 32]
    gen = cx10_make_generator_stack(channels)
    assert isinstance(gen, nn.Sequential), f'outer must be nn.Sequential, got {type(gen).__name__}'

    blocks = list(gen.children())
    assert len(blocks) == 3, f'expected 3 child blocks; got {len(blocks)}'

    # Case A: each block is itself a Sequential of (ConvT, BN, ReLU).
    for i, blk in enumerate(blocks):
        assert isinstance(blk, nn.Sequential), f'block {i} must be nn.Sequential, got {type(blk).__name__}'
        layers = list(blk.children())
        assert len(layers) == 3, f'block {i} should have 3 layers; got {len(layers)}'
        assert isinstance(layers[0], nn.ConvTranspose2d), f'block {i} layer 0 not ConvT'
        assert isinstance(layers[1], nn.BatchNorm2d), f'block {i} layer 1 not BatchNorm2d'
        assert isinstance(layers[2], nn.ReLU), f'block {i} layer 2 not ReLU'
        ct = layers[0]
        assert ct.in_channels == channels[i] and ct.out_channels == channels[i + 1], (
            f'block {i} channel chain wrong: in={ct.in_channels} (want {channels[i]}), '
            f'out={ct.out_channels} (want {channels[i + 1]})'
        )
        assert ct.kernel_size == (4, 4), f'block {i} kernel should be 4, got {ct.kernel_size}'
        assert ct.stride == (2, 2), f'block {i} stride should be 2, got {ct.stride}'
        assert ct.padding == (1, 1), f'block {i} padding should be 1, got {ct.padding}'
        assert ct.bias is None, f'block {i} ConvT must have bias=False'
        bn = layers[1]
        assert bn.num_features == channels[i + 1], (
            f'block {i} BatchNorm num_features should match ConvT out, got {bn.num_features}'
        )

    # Case B: spatial-doubling end to end.
    gen.eval()  # BN in eval avoids needing big batch.
    x = t.randn(2, channels[0], 4, 4)
    out = gen(x)
    assert out.shape == (2, channels[-1], 32, 32), (
        f'expected output shape (2, {channels[-1]}, 32, 32) after 3 doublings; got {tuple(out.shape)}'
    )

    # Case C: total parameter count proves all 3 blocks were registered (not just kept locally).
    params = list(gen.parameters())
    # Each block contributes: ConvT weight + BN weight + BN bias = 3 params (ConvT bias=False).
    # 3 blocks * 3 = 9 params.
    assert len(params) == 9, (
        f'expected 9 params (3 blocks * 3 each: ConvT.weight, BN.weight, BN.bias); got {len(params)}'
    )
    _dd_passed.add('cx10')

_test_cx10()

<details><summary>Show solution — cx10</summary>

```python
def cx10_make_generator_stack(channels):
    # Atom A (convtranspose-bn-activation-block): the reusable inner block.
    def block(c_in, c_out):
        return nn.Sequential(
            nn.ConvTranspose2d(c_in, c_out, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(c_out),
            nn.ReLU(inplace=True),
        )

    # Atom B (module-composition): wrap the three blocks in an outer Sequential
    # so they register as named children (block.0, block.1, block.2).
    return nn.Sequential(
        block(channels[0], channels[1]),
        block(channels[1], channels[2]),
        block(channels[2], channels[3]),
    )
```

DCGAN's classic generator uses 4 such blocks (4x4 -> 64x64). The factory pattern (`block(c_in, c_out)`) lets you generate the stack from a channel list of any length. Sequential-of-Sequentials is fully supported by PyTorch — the outer's `.parameters()` recursively collects from all nested blocks.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx10'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx10',
        'subtopics': ["PyTorch: Module composition", "GAN: ConvT+BN+Activation block"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()